# 04 · 因子合成

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
重算因子相关矩阵、三种权重方案与合成前后 IC，对照 `combine.json`。

In [ ]:
import pandas as pd, numpy as np, json
factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])
FACTORS = ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]

# ---- 相关矩阵（逐期平均）----
corr_pool = []
for _, g in factors.groupby("date"):
    z = g[[f"z_{f}" for f in FACTORS]].dropna()
    if len(z) >= 50: corr_pool.append(z.corr().values)
corr = np.nanmean(np.array(corr_pool), axis=0)
print("MOM60 × REV5 相关:", round(corr[2, 3], 3), "（动量与反转天然负相关）")

# ---- IC 向量与三种权重 ----
def ic_means(factors):
    out = {}
    for f in FACTORS:
        ics = []
        for _, g in factors.groupby("date"):
            x, y = g["z_" + f], g["next_return"]
            m = x.notna() & y.notna()
            if m.sum() >= 10: ics.append(np.corrcoef(x[m], y[m])[0, 1])
        a = np.array(ics); out[f] = dict(mean=a.mean(), icir=a.mean() / a.std())
    return out

icm = ic_means(factors)
ic_vec = np.array([icm[f]["mean"] for f in FACTORS])
icir = np.nan_to_num(np.array([icm[f]["icir"] for f in FACTORS]))
w_equal = np.full(8, 1 / 8)
w_ic = ic_vec / ic_vec.sum()
w_ir = np.clip(icir, 0, None); w_ir = w_ir / w_ir.sum() if w_ir.sum() > 0 else w_equal

# ---- 合成因子 IC ----
def comp_ic(w):
    out = []
    for _, g in factors.groupby("date"):
        z = g[[f"z_{f}" for f in FACTORS]].fillna(0.0).values @ w
        y = g["next_return"]
        m = y.notna() & np.isfinite(z)
        if m.sum() >= 20: out.append(np.corrcoef(z[m], y[m])[0, 1])
    return float(np.mean(out))

print("等权合成 IC:", round(comp_ic(w_equal), 4), "（高于任何单因子——分散化红利）")
print("IC 加权合成 IC:", round(comp_ic(w_ic), 4))
print("IR 加权合成 IC:", round(comp_ic(w_ir), 4))

In [ ]:
# ---- 对照 combine.json ----
ref = json.load(open("data/combine.json", encoding="utf-8"))
ok_corr = np.allclose(corr, np.array(ref["corr"]), atol=1e-4)
ok_w = (np.allclose(w_ic, ref["weights"]["ic"], atol=1e-3)
        and np.allclose(w_ir, ref["weights"]["ir"], atol=1e-3))
ok_ic = abs(comp_ic(w_equal) - ref["ic_compare"]["equal"]) < 1e-3
print("相关矩阵对照:", "PASS" if ok_corr else "FAIL")
print("权重对照:", "PASS" if ok_w else "FAIL")
print("合成 IC 对照:", "PASS" if ok_ic else "FAIL")
assert ok_corr and ok_w and ok_ic